In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark
import statsmodels.api as sm


1.  Build any model of your choice with tunable hyperparameters

_I will use random forest regression to predict the driver points scored in the sprint race. My features are: grid position and number of laps. I will have to load the results dataset for the analysis._

In [0]:
# Load the results data
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

# Inspect the data
display(df_results)

Perform a train/test split

In [0]:
# Change variable names to X, y to prepare for train/test split
df = df_results.toPandas()
y = df["points"]
X = df[["grid", "laps"]]

# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Build a Random Forest Regression Model

In [0]:
# Random Forest Regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

params = {"n_estimators": 200, "max_depth": 5, "random_state": 42}
rf = RandomForestRegressor(**params)
rf.fit(X_train, y_train)
predictions = rf.predict(X_test)

# Create metrics to evaluate model
mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print("  mse: {}".format(mse))
print("  mae: {}".format(mae))
print("  R2: {}".format(r2))

# Create feature importance
feature_importances = pd.DataFrame(rf.feature_importances_, index=X_train.columns, columns=["importance"])
feature_importances.sort_values("importance", ascending=False)
print(feature_importances)

# Create plot
fig, ax = plt.subplots()

sns.residplot(x=predictions, y=y_test.astype(float))
plt.xlabel("Predicted values for Driver Points")
plt.ylabel("Residual")
plt.title("Residual Plot")

2. Create an experiment setup where - for each run - you log:
- the hyperparameters used in the model
- the model itself
- every possible metric from the model you chose
- at least two artifacts (plots, or csv files)

In [0]:
import mlflow.sklearn
import tempfile
import os

with mlflow.start_run(run_name="Basic RF Experiment") as run:
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-model")

  # Log params
  [mlflow.log_param(param, value) for param, value in params.items()]
  
  # Log metrics
  mlflow.log_metric("mse", mse)
  mlflow.log_metric("mae", mae)  
  mlflow.log_metric("r2", r2)  
  
  # Log importances using a temporary file
  temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
  temp_name = temp.name
  try:
      feature_importances.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")

  finally:
      temp.close() # Delete the temp file
  
  # Log residuals using a temporary file
  temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
  temp_name = temp.name
  try:
    fig.savefig(temp_name)
    mlflow.log_artifact(temp_name, "residuals.png")
  finally:
    temp.close() # Delete the temp file
    
  display(fig)
  
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

3. Track your MLFlow experiment and run at least 10 experiments with different parameters each

In [0]:
def log_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):
  import os
  import matplotlib.pyplot as plt
  import mlflow.sklearn
  import seaborn as sns
  from sklearn.ensemble import RandomForestRegressor
  from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
  import tempfile

  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)

    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-model")

    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]

    # Create metrics
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    print("  mse: {}".format(mse))
    print("  mae: {}".format(mae))
    print("  R2: {}".format(r2))

    # Log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)  
    mlflow.log_metric("r2", r2)  
    
    # Create feature importance
    feature_importances = pd.DataFrame(rf.feature_importances_, index=X_train.columns, columns=["importance"])
    feature_importances.sort_values("importance", ascending=False)
    print(feature_importances)
    
    # Log importances using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      feature_importances.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")
    finally:
      temp.close() # Delete the temp file
    
    # Create plot
    fig, ax = plt.subplots()

    sns.residplot(x=predictions, y=y_test.astype(float), lowess=True)
    plt.xlabel("Predicted values for Driver Points")
    plt.ylabel("Residual")
    plt.title("Residual Plot")

    # Log residuals using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "residuals.png")
    finally:
      temp.close() # Delete the temp file
      
    display(fig)
    return run.info.run_id

In [0]:
params = {
  "n_estimators": 100,
  "max_depth": 10,
  "random_state": 46
}

log_rf(experimentID, "Second Run", params, X_train, X_test, y_train, y_test)